In [ ]:
!apt update -y
!apt install -y git wget
!pip install -U huggingface_hub accelerate transformers safetensors

In [ ]:
%cd /tmp

!git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /tmp/ComfyUI

In [ ]:
!mkdir -p /tmp/ComfyUI/models/diffusion_models
!mkdir -p /tmp/ComfyUI/models/text_encoders
!mkdir -p /tmp/ComfyUI/models/vae

In [ ]:
!HF_XET_HIGH_PERFORMANCE=1 \
hf download Comfy-Org/MiniMax-H3 \
  --repo-type model \
  --revision main \
  --include "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors" \
  --local-dir /tmp/ComfyUI/models

In [ ]:
!HF_XET_HIGH_PERFORMANCE=1 \
hf download Comfy-Org/MiniMax-H3 \
  --repo-type model \
  --revision main \
  --include "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" \
  --local-dir /tmp/ComfyUI/models

In [ ]:
!HF_XET_HIGH_PERFORMANCE=1 \
hf download Comfy-Org/MiniMax-H3 \
  --repo-type model \
  --revision main \
  --include "vae/minimax_h3_video_vae_fp16.safetensors" \
  --local-dir /tmp/ComfyUI/models

In [ ]:
!cd /tmp/ComfyUI && \
wget -c \
"https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors" \
-P models/vae/

In [ ]:
!echo "===== Diffusion Models =====" && \
ls -lh /tmp/ComfyUI/models/diffusion_models && \
echo "" && \
echo "===== Text Encoders =====" && \
ls -lh /tmp/ComfyUI/models/text_encoders && \
echo "" && \
echo "===== Video VAE =====" && \
ls -lh /tmp/ComfyUI/models/vae && \
echo "" && \
echo "===== Audio Models =====" && \
ls -lh /tmp/ComfyUI/models/audio_vae

In [ ]:
%cd /tmp/ComfyUI

!pip install -r requirements.txt

In [ ]:
pip install -U --pre comfyui-manager

In [ ]:
import os
import subprocess
import time

COMFY_DIR = "/tmp/ComfyUI"

# ============================================================
# 1. STOP ALL EXISTING COMFYUI PROCESSES
# ============================================================

subprocess.run(
    "pkill -9 -f '/tmp/ComfyUI/main.py'",
    shell=True,
    stderr=subprocess.DEVNULL
)

# Free ports
subprocess.run(
    "fuser -k 8188/tcp 8189/tcp 2>/dev/null",
    shell=True,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

# ============================================================
# 2. GPU 0 → PORT 8188 → OWN DATABASE
# ============================================================

env0 = os.environ.copy()
env0["CUDA_VISIBLE_DEVICES"] = "0"

log0 = open("/tmp/comfyui_gpu0.log", "w")

process_gpu0 = subprocess.Popen(
    [
        "python",
        "main.py",
        "--listen", "0.0.0.0",
        "--port", "8188",
        "--database-url",
        "sqlite:////tmp/comfy_gpu0.db",
    ],
    cwd=COMFY_DIR,
    env=env0,
    stdout=log0,
    stderr=subprocess.STDOUT,
)

print("GPU 0 ComfyUI started")
print("PID:", process_gpu0.pid)
print("Port: 8188")
print("Database: /tmp/comfy_gpu0.db")

# ============================================================
# 3. GPU 1 → PORT 8189 → OWN DATABASE
# ============================================================

env1 = os.environ.copy()
env1["CUDA_VISIBLE_DEVICES"] = "1"

log1 = open("/tmp/comfyui_gpu1.log", "w")

process_gpu1 = subprocess.Popen(
    [
        "python",
        "main.py",
        "--listen", "0.0.0.0",
        "--port", "8189",
        "--database-url",
        "sqlite:////tmp/comfy_gpu1.db",
    ],
    cwd=COMFY_DIR,
    env=env1,
    stdout=log1,
    stderr=subprocess.STDOUT,
)

print("GPU 1 ComfyUI started")
print("PID:", process_gpu1.pid)
print("Port: 8189")
print("Database: /tmp/comfy_gpu1.db")

# ============================================================
# 4. WAIT FOR SERVERS
# ============================================================

time.sleep(50)

# ============================================================
# 5. CHECK PROCESSES
# ============================================================

print("\n========== PROCESSES ==========")

subprocess.run(
    "ps aux | grep '[m]ain.py'",
    shell=True
)

# ============================================================
# 6. CHECK PORTS
# ============================================================

print("\n========== PORTS ==========")

subprocess.run(
    "ss -ltnp | grep -E ':8188|:8189' || echo 'Ports not listening'",
    shell=True
)

# ============================================================
# 7. CHECK GPU MEMORY
# ============================================================

print("\n========== GPU ==========")

subprocess.run([
    "nvidia-smi",
    "--query-gpu=index,name,memory.used,memory.total",
    "--format=csv"
])

# ============================================================
# 8. CHECK LOGS FOR DATABASE ERRORS
# ============================================================

print("\n========== GPU 0 LOG ==========")

subprocess.run(
    "tail -30 /tmp/comfyui_gpu0.log",
    shell=True
)

print("\n========== GPU 1 LOG ==========")

subprocess.run(
    "tail -30 /tmp/comfyui_gpu1.log",
    shell=True
)

In [ ]:
!tail -100 /tmp/comfyui_gpu0.log

In [ ]:
!tail -100 /tmp/comfyui_gpu1.log

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
!ls -lh /tmp/cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import subprocess
import os

subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/tmp/cloudflared"
], check=True)

os.chmod("/tmp/cloudflared", 0o755)

print("Cloudflared installed.")
print(subprocess.check_output(
    ["/tmp/cloudflared", "--version"],
    text=True
))

In [ ]:
import subprocess
import os
import time
import re
import socket
import signal
import urllib.request
import urllib.error

# ============================================================
# HARD-CODED CONFIGURATION
# ============================================================

PORT_GPU0 = 8188
PORT_GPU1 = 8189

HOST_GPU0 = "127.0.0.1"
HOST_GPU1 = "127.0.0.1"

LOG_GPU0 = "/tmp/cloudflared_8188.log"
LOG_GPU1 = "/tmp/cloudflared_8189.log"

CLOUDFLARED = "cloudflared"

METRICS_GPU0 = 18188
METRICS_GPU1 = 18189

URL_TIMEOUT = 60

# Give newly created trycloudflare DNS time to appear
DNS_WAIT_TIMEOUT = 120

# How often to retry DNS
DNS_RETRY_INTERVAL = 3

# HTTP test timeout
HTTP_TIMEOUT = 15


# ============================================================
# BASIC HELPERS
# ============================================================

def port_is_open(host, port, timeout=2):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(timeout)

    try:
        sock.connect((host, port))
        return True
    except (socket.timeout, ConnectionRefusedError, OSError):
        return False
    finally:
        sock.close()


def wait_for_port(host, port, timeout=30):
    start = time.time()

    while time.time() - start < timeout:
        if port_is_open(host, port):
            return True

        time.sleep(1)

    return False


def command_exists(command):
    try:
        result = subprocess.run(
            ["bash", "-lc", f"command -v {command}"],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
            check=False
        )

        return result.returncode == 0

    except Exception:
        return False


# ============================================================
# STOP OLD CLOUDFLARED
# ============================================================

def kill_cloudflared_processes():

    print("\nStopping old cloudflared processes...")

    try:

        result = subprocess.run(
            ["pgrep", "-x", "cloudflared"],
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
            check=False
        )

        pids = []

        for line in result.stdout.splitlines():

            line = line.strip()

            if line.isdigit():
                pids.append(int(line))

        for pid in pids:

            try:
                os.kill(pid, signal.SIGKILL)
                print(f"  Killed cloudflared PID {pid}")

            except ProcessLookupError:
                pass

            except PermissionError:
                print(f"  Permission denied for PID {pid}")

    except Exception as e:

        print(f"  Cleanup warning: {e}")

    time.sleep(2)


# ============================================================
# DELETE LOGS
# ============================================================

def delete_old_logs():

    for logfile in (
        LOG_GPU0,
        LOG_GPU1
    ):

        try:
            os.remove(logfile)

        except FileNotFoundError:
            pass

        except Exception as e:
            print(
                f"Could not remove {logfile}: {e}"
            )


# ============================================================
# START TUNNEL
# ============================================================

def start_tunnel(
    host,
    port,
    logfile,
    metrics_port
):

    print("\n" + "-" * 80)

    print("Starting Cloudflare Quick Tunnel")
    print(f"LOCAL   : http://{host}:{port}")
    print(f"LOG     : {logfile}")
    print(f"METRICS : 127.0.0.1:{metrics_port}")

    print("-" * 80)

    log = open(
        logfile,
        "w",
        buffering=1
    )

    cmd = [
        CLOUDFLARED,
        "tunnel",
        "--url",
        f"http://{host}:{port}",
        "--no-autoupdate",
        "--metrics",
        f"127.0.0.1:{metrics_port}"
    ]

    process = subprocess.Popen(
        cmd,
        stdout=log,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
        start_new_session=True
    )

    return process


# ============================================================
# EXTRACT CLOUDFLARE URL
# ============================================================

def get_cloudflare_url(
    logfile,
    timeout=URL_TIMEOUT
):

    pattern = re.compile(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
    )

    start = time.time()

    while time.time() - start < timeout:

        try:

            with open(
                logfile,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                text = f.read()

            matches = pattern.findall(text)

            if matches:
                return matches[-1]

        except FileNotFoundError:
            pass

        time.sleep(1)

    return None


# ============================================================
# DNS TEST
# ============================================================

def resolve_hostname(hostname):

    try:

        results = socket.getaddrinfo(
            hostname,
            443,
            type=socket.SOCK_STREAM
        )

        addresses = []

        for result in results:

            sockaddr = result[4]

            if sockaddr:

                ip = sockaddr[0]

                if ip not in addresses:
                    addresses.append(ip)

        return addresses

    except socket.gaierror:
        return []


# ============================================================
# WAIT FOR DNS
# ============================================================

def wait_for_dns(
    url,
    timeout=DNS_WAIT_TIMEOUT
):

    if not url:
        return False, []

    hostname = url.split(
        "://",
        1
    )[-1].split(
        "/",
        1
    )[0]

    print(
        f"\nWaiting for DNS: {hostname}"
    )

    start = time.time()

    while time.time() - start < timeout:

        addresses = resolve_hostname(
            hostname
        )

        if addresses:

            print(
                f"✅ DNS resolved: "
                f"{hostname} → "
                f"{', '.join(addresses)}"
            )

            return True, addresses

        elapsed = int(
            time.time() - start
        )

        print(
            f"  DNS not ready yet "
            f"({elapsed}s/{timeout}s)"
        )

        time.sleep(
            DNS_RETRY_INTERVAL
        )

    print(
        f"❌ DNS did not resolve: "
        f"{hostname}"
    )

    return False, []


# ============================================================
# PUBLIC HTTP TEST
# ============================================================

def test_public_url(
    url,
    timeout=HTTP_TIMEOUT
):

    if not url:

        return (
            False,
            None,
            "NO URL"
        )

    try:

        request = urllib.request.Request(
            url,
            method="GET",
            headers={
                "User-Agent":
                    "Mozilla/5.0"
            }
        )

        with urllib.request.urlopen(
            request,
            timeout=timeout
        ) as response:

            status = response.status

            body = response.read(
                500
            ).decode(
                "utf-8",
                errors="ignore"
            )

            return (
                True,
                status,
                body
            )

    except urllib.error.HTTPError as e:

        try:

            body = e.read(
                500
            ).decode(
                "utf-8",
                errors="ignore"
            )

        except Exception:

            body = ""

        return (
            False,
            e.code,
            body
        )

    except Exception as e:

        return (
            False,
            None,
            str(e)
        )


# ============================================================
# TEST ONE COMPLETE TUNNEL
# ============================================================

def validate_tunnel(
    name,
    process,
    url
):

    print("\n" + "=" * 80)
    print(f"VALIDATING {name}")
    print("=" * 80)

    if not process:

        print("❌ No process")
        return False

    if process.poll() is not None:

        print(
            f"❌ cloudflared stopped "
            f"with code {process.returncode}"
        )

        return False

    print("✅ cloudflared process is running")

    if not url:

        print("❌ No Cloudflare URL")
        return False

    print(f"URL: {url}")

    # --------------------------------------------------------
    # DNS
    # --------------------------------------------------------

    dns_ok, addresses = wait_for_dns(
        url
    )

    if not dns_ok:

        print(
            "❌ DNS is not ready"
        )

        return False

    # --------------------------------------------------------
    # HTTP
    # --------------------------------------------------------

    print(
        "\nTesting public HTTP..."
    )

    ok, status, body = test_public_url(
        url
    )

    if ok:

        print(
            f"✅ PUBLIC HTTP OK "
            f"(HTTP {status})"
        )

        return True

    print(
        f"❌ PUBLIC HTTP FAILED"
    )

    print(
        f"Status: {status}"
    )

    print(
        f"Response: {body[:500]}"
    )

    return False


# ============================================================
# LOG TAIL
# ============================================================

def print_log_tail(
    logfile,
    lines=30
):

    print("\n" + "-" * 80)
    print(
        f"LAST {lines} LOG LINES: "
        f"{logfile}"
    )
    print("-" * 80)

    try:

        with open(
            logfile,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            content = f.readlines()

        for line in content[-lines:]:

            print(
                line.rstrip()
            )

    except FileNotFoundError:

        print(
            "Log file not found."
        )


# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print(
    "       COMFYUI + CLOUDFLARE QUICK TUNNELS"
)
print("=" * 80)


# ============================================================
# CHECK CLOUDFLARED
# ============================================================

print(
    "\nChecking cloudflared..."
)

if not command_exists(
    CLOUDFLARED
):

    print(
        "❌ cloudflared not found"
    )

    raise SystemExit(1)

print(
    "✅ cloudflared found"
)

version = subprocess.run(
    [
        CLOUDFLARED,
        "--version"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False
)

print(
    version.stdout.strip()
)


# ============================================================
# CHECK COMFYUI
# ============================================================

print("\n" + "=" * 80)
print("CHECKING COMFYUI PORTS")
print("=" * 80)

gpu0_live = wait_for_port(
    HOST_GPU0,
    PORT_GPU0,
    10
)

gpu1_live = wait_for_port(
    HOST_GPU1,
    PORT_GPU1,
    10
)

print(
    f"GPU 0 | "
    f"{HOST_GPU0}:{PORT_GPU0} | "
    f"{'✅ LIVE' if gpu0_live else '❌ OFFLINE'}"
)

print(
    f"GPU 1 | "
    f"{HOST_GPU1}:{PORT_GPU1} | "
    f"{'✅ LIVE' if gpu1_live else '❌ OFFLINE'}"
)

if not gpu0_live or not gpu1_live:

    print(
        "\n❌ Both ComfyUI ports "
        "must be running."
    )

    raise SystemExit(1)

print(
    "\n✅ Both ComfyUI ports are LIVE"
)


# ============================================================
# CLEANUP
# ============================================================

kill_cloudflared_processes()

delete_old_logs()


# ============================================================
# START GPU 0
# ============================================================

tunnel_gpu0 = start_tunnel(
    HOST_GPU0,
    PORT_GPU0,
    LOG_GPU0,
    METRICS_GPU0
)

time.sleep(2)


# ============================================================
# START GPU 1
# ============================================================

tunnel_gpu1 = start_tunnel(
    HOST_GPU1,
    PORT_GPU1,
    LOG_GPU1,
    METRICS_GPU1
)


# ============================================================
# GET URLS
# ============================================================

print("\n" + "=" * 80)
print(
    "WAITING FOR CLOUDFLARE PUBLIC URLS"
)
print("=" * 80)

url_gpu0 = get_cloudflare_url(
    LOG_GPU0
)

url_gpu1 = get_cloudflare_url(
    LOG_GPU1
)


# ============================================================
# SHOW URLS
# ============================================================

print("\nGPU 0 URL:")
print(
    url_gpu0 or "❌ NOT FOUND"
)

print("\nGPU 1 URL:")
print(
    url_gpu1 or "❌ NOT FOUND"
)


# ============================================================
# VALIDATE GPU 0
# ============================================================

gpu0_ok = validate_tunnel(
    "GPU 0",
    tunnel_gpu0,
    url_gpu0
)


# ============================================================
# VALIDATE GPU 1
# ============================================================

gpu1_ok = validate_tunnel(
    "GPU 1",
    tunnel_gpu1,
    url_gpu1
)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n" + "=" * 80)
print("FINAL RESULT")
print("=" * 80)

print(
    f"GPU 0 | "
    f"{'✅ READY' if gpu0_ok else '❌ NOT READY'}"
)

print(
    f"GPU 1 | "
    f"{'✅ READY' if gpu1_ok else '❌ NOT READY'}"
)

if url_gpu0:
    print(
        f"\nGPU 0 → {url_gpu0}"
    )

if url_gpu1:
    print(
        f"GPU 1 → {url_gpu1}"
    )

if gpu0_ok and gpu1_ok:

    print(
        "\n✅ BOTH PUBLIC TUNNELS ARE READY"
    )

else:

    print(
        "\n⚠️ ONE OR MORE TUNNELS "
        "ARE NOT READY"
    )

    if not gpu0_ok:
        print_log_tail(
            LOG_GPU0
        )

    if not gpu1_ok:
        print_log_tail(
            LOG_GPU1
        )

print("=" * 80)

In [ ]:
!cat /tmp/cloudflare_8188.log

In [ ]:
!cat /tmp/cloudflare_8189.log

In [ ]:
!sleep 100000